# 01 — Geo source profiling: communes + RPG (Tarn-et-Garonne, dept 82)

**Purpose:** Before building the pipeline, verify what the actual API returns.

- **Shape:** Which fields are returned, with what data types and geometry types?
- **Volume:** How many parcels are contained in a single tile, and what is the estimated volume
            when scaling to the entire department? 
- **Quality:** Invalid geometries, missing values, and discrepancies between declared and calculated areas.

Scope: All communes (to generate the departmental boundary), 3 sample communes, and 1 RPG 5km tile.
This notebook is for exploratory analysis. Production code belongs in `src/hive_platform/ingest/`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from shapely.geometry import box, mapping
from urllib3.util.retry import Retry

DEPT_CODE = "82"
RPG_YEAR = 2024
CRS_WGS84 = "EPSG:4326"
CRS_L93 = "EPSG:2154"
GEO_API = "https://geo.api.gouv.fr"
RPG_API = "https://apicarto.ign.fr/api/rpg/v2"
TILE_SIZE_M = 5000
SAMPLE_COMMUNE = "Moissac"   # Orchard-dense area -> Select the sample tile here

# VSCode uses the notebook folder as the working directory, so paths should be resolved relative to the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLES_DIR = ROOT / "data" / "samples"

pd.set_option("display.max_columns", 30)
print("geopandas", gpd.__version__, "| root:", ROOT)

In [ ]:
def make_session() -> requests.Session:
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=1.0, status_forcelist=(429, 500, 502, 503, 504))
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s


def get_json(session, url, params, timeout=120) -> dict:
    r = session.get(url, params=params, timeout=timeout)
    if not r.ok:
        print("HTTP", r.status_code, r.text[:1000])   # See first error messsage
        r.raise_for_status()
    print(f"GET {r.url[:120]}...  ->  {r.status_code}, {len(r.content) / 1024:.0f} KB")
    return r.json()


session = make_session()

## STEP 1 — Commune API Original JSON

In [ ]:
communes_raw = get_json(
    session,
    f"{GEO_API}/departements/{DEPT_CODE}/communes",
    {"format": "geojson", "geometry": "contour", "fields": "code,nom,population"},
    timeout=60,
)

print("top-level keys:", list(communes_raw.keys()))
print("commune number:", len(communes_raw["features"]))

f0 = communes_raw["features"][0]
print("\n첫 feature properties:")
print(json.dumps(f0["properties"], indent=2, ensure_ascii=False))
print("geometry type:", f0["geometry"]["type"])
# 좌표 전체를 찍는 대신 꼭짓점 수만 본다 (geometry 무게 감 잡기)
ring = f0["geometry"]["coordinates"][0]
ring = ring[0] if f0["geometry"]["type"] == "MultiPolygon" else ring
print("첫 ring 꼭짓점 수:", len(ring), "| 첫 좌표 (lon, lat):", ring[0])

## STEP 2 — Commune GeoDataFrame

In [ ]:
communes = gpd.GeoDataFrame.from_features(communes_raw["features"], crs=CRS_WGS84)

print("shape:", communes.shape)
print("CRS:", communes.crs)
print("\ngeometry type:\n", communes.geom_type.value_counts())
print("\ndtypes:\n", communes.dtypes)
print("\nnull number:\n", communes.isna().sum())
print("\npopulation distribution:\n", communes["population"].describe())

In [ ]:
sample_communes = communes.head(3).copy()
display(sample_communes.drop(columns="geometry"))

# Area must always be calculated in a metric coordinate system(Lambert-93)
sample_communes["area_km2"] = sample_communes.to_crs(CRS_L93).area / 1e6
display(sample_communes[["code", "nom", "population", "area_km2"]])

## STEP 3 — Merge Communes to create the department boundary (dissolve)

In [ ]:
boundary = communes[["geometry"]].dissolve().reset_index(drop=True)
boundary["dept_code"] = DEPT_CODE

area_km2 = boundary.to_crs(CRS_L93).area.iloc[0] / 1e6
print("boundary geometry type:", boundary.geom_type.iloc[0])
print(f"department area: {area_km2:,.0f} km²   (sanity check: official area is approximately 3,700 km²)")

## STEP 4 — Create 5 km Tiles and Select a Sample Tile

In [ ]:
b93 = boundary.to_crs(CRS_L93)
minx, miny, maxx, maxy = b93.total_bounds
print("bounds (L93, m):", [round(v) for v in (minx, miny, maxx, maxy)])

grid = gpd.GeoDataFrame(
    geometry=[
        box(x, y, x + TILE_SIZE_M, y + TILE_SIZE_M)
        for x in np.arange(minx, maxx, TILE_SIZE_M)
        for y in np.arange(miny, maxy, TILE_SIZE_M)
    ],
    crs=CRS_L93,
)
bbox_tiles = len(grid)
grid = grid[grid.intersects(b93.geometry.iloc[0])].reset_index(drop=True)
grid["tile_id"] = grid.index
print(f"bbox 타일 {bbox_tiles}개 -> 도와 겹치는 타일 {len(grid)}개")

# The first tile (tiles.iloc[0]) corresponds to the lower-left corner of the bbox,
# so it may be mostly outside the department. Therefore, select the tile that contains the centroid of the orchard-dense commune.
anchor = communes[communes["nom"] == SAMPLE_COMMUNE].to_crs(CRS_L93).geometry.iloc[0].centroid
sample_tile_l93 = grid[grid.intersects(anchor)].iloc[0]
inside_ratio = sample_tile_l93.geometry.intersection(b93.geometry.iloc[0]).area / sample_tile_l93.geometry.area
print(f"Sample tile tile_id={sample_tile_l93.tile_id} ({SAMPLE_COMMUNE}), {inside_ratio:.0%} of tile is within the department")

tiles = grid.to_crs(CRS_WGS84)
sample_tile = tiles.loc[tiles.tile_id == sample_tile_l93.tile_id].geometry.iloc[0]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
boundary.to_crs(CRS_L93).plot(ax=ax, color="#eeeeee", edgecolor="#555555")
grid.boundary.plot(ax=ax, color="#bbbbbb", linewidth=0.4)
gpd.GeoSeries([sample_tile_l93.geometry], crs=CRS_L93).plot(ax=ax, color="tab:orange", alpha=0.7)
ax.set_title(f"Dept {DEPT_CODE}: {len(grid)} tiles of {TILE_SIZE_M // 1000} km, sample tile = {SAMPLE_COMMUNE}")
ax.set_axis_off()
plt.show()

## STEP 5 — RPG API: Retrieve all data from a single tile (including Pagination)

In [ ]:
def fetch_rpg_tile(session, geom, year, page_size=1000):
    pages, start = [], 0
    while True:
        params = {"annee": year, "geom": json.dumps(mapping(geom)), "_limit": page_size, "_start": start}
        data = get_json(session, RPG_API, params)
        pages.append(data)
        if len(data.get("features", [])) < page_size:
            return pages
        start += page_size


pages = fetch_rpg_tile(session, sample_tile, RPG_YEAR)
rpg_raw = pages[0]
features = [f for p in pages for f in p["features"]]

print("top-level keys:", [k for k in rpg_raw.keys() if k != "features"])
print("numberMatched (total count):", rpg_raw.get("numberMatched"), "| page count:", len(pages), "| received features:", len(features))
print("\nfirst feature properties:")
print(json.dumps(features[0]["properties"], indent=2, ensure_ascii=False))
print("geometry type:", features[0]["geometry"]["type"])

In [ ]:
# Get a rough sense of the volume: use this single tile to extrapolate to the entire department.
tile_kb = len(json.dumps({"features": features}).encode()) / 1024
n = len(features)
print(f"this tile: {n:,}, GeoJSON {tile_kb:,.0f} KB")
print(f"department-wide estimate (x {len(grid)} tiles): ~{n * len(grid):,}, ~{tile_kb * len(grid) / 1024:,.0f} MB")
print("Note: This is based on an orchard-dense tile, so edge tiles contain fewer parcels, so this is closer to an upper bound estimate. Parcels crossing tile boundaries will be duplicated.")

## STEP 6 — RPG GeoDataFrame

In [ ]:
rpg = gpd.GeoDataFrame.from_features(features, crs=CRS_WGS84)

print("shape:", rpg.shape)
print("\ndtypes:\n", rpg.dtypes)
print("\nnull number:\n", rpg.isna().sum())
print("\nid_parcel duplicates:", rpg["id_parcel"].duplicated().sum())
display(rpg.drop(columns="geometry").head())

In [ ]:
# What types of crops are present: input for the next decision (defining the parcels that require pollination)
print("cat_cult_p (TA annual crops / CP permanent crops / PP permanent grassland / SB woodland):")
print(rpg["cat_cult_p"].value_counts(dropna=False))
print("\ncode_group:")
print(rpg["code_group"].value_counts().head(15))
print("\ncode_cultu top 15:")
print(rpg["code_cultu"].value_counts().head(15))

## STEP 7 — Geometry validation

In [ ]:
print("Invalid before fix:", (~rpg.is_valid).sum())
print("Geometry types (before):\n", rpg.geom_type.value_counts())

rpg["geometry"] = rpg.geometry.make_valid()

print("\nInvalid after fix:", (~rpg.is_valid).sum())
print("Geometry types (after):\n", rpg.geom_type.value_counts())

# make_valid can produce GeometryCollections → decide here whether a later step to extract polygons is needed

## STEP 8 — 면적 검증: 신고 면적(surf_parc) vs 계산 면적

In [ ]:
rpg["area_ha_calc"] = rpg.to_crs(CRS_L93).area / 10_000
rpg["area_diff_pct"] = (rpg["area_ha_calc"] - rpg["surf_parc"]) / rpg["surf_parc"] * 100

print("surf_parc (ha) 분포:\n", rpg["surf_parc"].describe())
print("\n계산 면적과의 차이 (%):\n", rpg["area_diff_pct"].describe())
print("\n|차이| > 5% 필지 수:", (rpg["area_diff_pct"].abs() > 5).sum())

## STEP 9 — 도 경계로 공간 필터링

In [ ]:
before = len(rpg)
rpg_in = rpg[rpg.intersects(boundary.geometry.iloc[0])].copy()
print(f"필터 전 {before} -> 후 {len(rpg_in)} (제거 {before - len(rpg_in)})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
r93 = rpg_in.to_crs(CRS_L93)
r93.plot(ax=ax, column="cat_cult_p", categorical=True, legend=True, cmap="tab10", linewidth=0)
r93[r93["cat_cult_p"] == "CP"].boundary.plot(ax=ax, color="black", linewidth=0.3)
gpd.GeoSeries([sample_tile_l93.geometry], crs=CRS_L93).boundary.plot(ax=ax, color="tab:orange")
ax.set_title(f"RPG {RPG_YEAR}, tile around {SAMPLE_COMMUNE} (outlined = permanent crops)")
ax.set_axis_off()
plt.show()

## STEP 10 — 작은 샘플을 `data/samples/`에 저장

나중에 네트워크 없이 도는 단위 테스트의 fixture로 쓴다. 용량을 작게 유지해서 커밋한다.

In [ ]:
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

communes_sample_path = SAMPLES_DIR / f"communes_{DEPT_CODE}_sample.geojson"
communes_sample = {"type": "FeatureCollection", "features": communes_raw["features"][:3]}
communes_sample_path.write_text(json.dumps(communes_sample, ensure_ascii=False), encoding="utf-8")

rpg_sample_path = SAMPLES_DIR / f"rpg_{DEPT_CODE}_{RPG_YEAR}_tile_sample.geojson"
rpg_sample = {k: v for k, v in rpg_raw.items() if k != "features"} | {"features": features[:50]}
rpg_sample_path.write_text(json.dumps(rpg_sample, ensure_ascii=False), encoding="utf-8")

for p in (communes_sample_path, rpg_sample_path):
    print(p.relative_to(ROOT), f"{p.stat().st_size / 1024:.0f} KB")

**## Observations (to be written based on the actual results)**

* **Shape:** Which RPG fields should be used in the pipeline, and which should be discarded?
* **Volume:** How many requests and how much data should we expect when collecting the entire department?
* **Quality:** What do the invalid geometry rate and area discrepancies tell us?
* **Decision needed:** How should we define “parcels requiring pollination”: by `cat_cult_p`, `code_group`, or `code_cultu`?
